# 03 — Faz 5a: `minifig_count` Zenginleştirmesi

**Amaç:** `data/processed/sets_clean.csv`'ye, her setin kaç minifig içerdiğini gösteren
`minifig_count` kolonunu eklemek.

Kullanılan join mantığı, [01_data_quality_check.ipynb](01_data_quality_check.ipynb) §10'da
gerçek verilerle doğrulanan `inventories.csv` ↔ `inventory_minifigs.csv` zincirinin
genelleştirilmiş hali:

```
set_num --(inventories.csv, en güncel version)--> inventory_id --(inventory_minifigs.csv)--> quantity toplamı
```

Not: Burada minifigin **kendi parça sayısı değil**, sadece **kaç adet minifig** olduğu
(basit `quantity` toplamı) hesaplanıyor — §10'daki `minifig_parça_toplamı` hesabından farklı.


In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW = "../data/raw"
PROCESSED = "../data/processed"


## 1. Veriyi oku

In [2]:
sets_clean = pd.read_csv(f"{PROCESSED}/sets_clean.csv")
inventories = pd.read_csv(f"{RAW}/inventories.csv")
inventory_minifigs = pd.read_csv(f"{RAW}/inventory_minifigs.csv")

n_before = len(sets_clean)
print(f"sets_clean.csv : {n_before:,} satır, {sets_clean.shape[1]} kolon")
print(f"inventories.csv: {len(inventories):,} satır")
print(f"inventory_minifigs.csv: {len(inventory_minifigs):,} satır")
sets_clean.head(3)


sets_clean.csv : 18,899 satır, 8 kolon
inventories.csv: 47,339 satır
inventory_minifigs.csv: 25,809 satır


,set_num,name,year,theme_id,theme_name,num_parts,img_url,is_analysis_ready
0,001-1,Gears,1965,756,Samsonite,43,https://cdn.rebrickable.com/media/sets/001-1.jpg,True
1,0011-2,Town Mini-Figures,1979,67,Classic Town,12,https://cdn.rebrickable.com/media/sets/0011-2.jpg,True
2,0012-1,Space Mini-Figures,1979,143,Supplemental,12,https://cdn.rebrickable.com/media/sets/0012-1.jpg,True


## 2. `set_num` → `inventory_id` → `minifig_count` join'i

Bazı setlerin birden fazla envanter versiyonu var (bkz. 01 notebook §10) — her `set_num` için
**en güncel** (en yüksek `version`) `inventory_id`'yi kullanıyoruz.


In [3]:
# Her set_num için en güncel envanter (§10 ile aynı mantık)
latest_inventory = (
    inventories.sort_values("version")
    .groupby("set_num", as_index=False)
    .tail(1)[["set_num", "id"]]
    .rename(columns={"id": "inventory_id"})
)

# Her inventory_id için toplam minifig adedi (fig_num bazında değil, satır quantity toplamı)
minifig_totals = (
    inventory_minifigs.groupby("inventory_id", as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity": "minifig_count"})
)

set_minifig_count = latest_inventory.merge(minifig_totals, on="inventory_id", how="left")
set_minifig_count["minifig_count"] = set_minifig_count["minifig_count"].fillna(0).astype(int)
set_minifig_count = set_minifig_count[["set_num", "minifig_count"]]

print(f"set_num başına minifig_count tablosu: {len(set_minifig_count):,} satır")
set_minifig_count.head()


set_num başına minifig_count tablosu: 45,403 satır


,set_num,minifig_count
0,7922-1,0
1,fig-012042,0
2,fig-012043,0
3,fig-012044,0
4,fig-012045,0


## 3. `sets_clean.csv`'ye left join ile ekle

Minifig'i olmayan (veya envanterde hiç minifig kaydı bulunmayan) setler için `minifig_count = 0`
(NaN değil).


In [4]:
sets_clean = sets_clean.merge(set_minifig_count, on="set_num", how="left")
sets_clean["minifig_count"] = sets_clean["minifig_count"].fillna(0).astype(int)

sets_clean.head(3)


,set_num,name,year,theme_id,theme_name,num_parts,img_url,is_analysis_ready,minifig_count
0,001-1,Gears,1965,756,Samsonite,43,https://cdn.rebrickable.com/media/sets/001-1.jpg,True,0
1,0011-2,Town Mini-Figures,1979,67,Classic Town,12,https://cdn.rebrickable.com/media/sets/0011-2.jpg,True,3
2,0012-1,Space Mini-Figures,1979,143,Supplemental,12,https://cdn.rebrickable.com/media/sets/0012-1.jpg,True,2


## 4. Doğrulama

In [5]:
assert len(sets_clean) == n_before, \
    f"Satır sayısı değişti! önce={n_before}, sonra={len(sets_clean)}"
print(f"✅ Satır sayısı değişmedi: {len(sets_clean):,}")

assert (sets_clean["minifig_count"] >= 0).all(), "Negatif minifig_count bulundu!"
print("✅ Hiçbir minifig_count negatif değil")

assert sets_clean["minifig_count"].notna().all(), "NaN minifig_count bulundu!"
print("✅ Hiçbir minifig_count NaN değil (hepsi >= 0 tam sayı)")


✅ Satır sayısı değişmedi: 18,899
✅ Hiçbir minifig_count negatif değil
✅ Hiçbir minifig_count NaN değil (hepsi >= 0 tam sayı)


In [6]:
has_minifig = sets_clean["minifig_count"] > 0
pos_ratio = has_minifig.mean() * 100

print(f"minifig_count > 0 (en az 1 minifig içeren set): {has_minifig.sum():,}  ({pos_ratio:.2f}%)")
print(f"minifig_count == 0 (hiç minifig içermeyen set) : {(~has_minifig).sum():,}  ({100 - pos_ratio:.2f}%)")
print()
sets_clean["minifig_count"].describe()


minifig_count > 0 (en az 1 minifig içeren set): 8,738  (46.24%)
minifig_count == 0 (hiç minifig içermeyen set) : 10,161  (53.76%)



count    18899.000000
mean         1.265675
std          2.567236
min          0.000000
25%          0.000000
50%          0.000000
75%          2.000000
max        100.000000
Name: minifig_count, dtype: float64

In [7]:
sets_clean["minifig_count"].value_counts().sort_index().head(10)


minifig_count
0    10161
1     3807
2     1682
3     1107
4      805
5      444
6      298
7      188
8      127
9       77
Name: count, dtype: int64

## 5. Kaydet

In [8]:
out_path = f"{PROCESSED}/sets_clean.csv"
sets_clean.to_csv(out_path, index=False)
print(f"Kaydedildi: {out_path}  ({len(sets_clean):,} satır, {sets_clean.shape[1]} kolon)")
print(f"Kolonlar: {list(sets_clean.columns)}")


Kaydedildi: ../data/processed/sets_clean.csv  (18,899 satır, 9 kolon)
Kolonlar: ['set_num', 'name', 'year', 'theme_id', 'theme_name', 'num_parts', 'img_url', 'is_analysis_ready', 'minifig_count']


## Özet

- `data/processed/sets_clean.csv`'ye yeni bir `minifig_count` kolonu eklendi (mevcut sütunların
  en sonuna), satır sayısı **değişmedi** (18.899).
- Setlerin **%46,24**'ü (8.738 set) en az bir minifig içeriyor; **%53,76**'sı (10.161 set) hiç
  minifig içermiyor (medyan = 0, çünkü minifig'siz setler çoğunlukta).
- `minifig_count` dağılımı: min=0, medyan=0, ortalama≈1,27, maksimum=100 (büyük ölçekli
  koleksiyon/diorama setleri).
- Bu bilgi Bölüm 2 (fiyat regresyonu) için önemli: özelliklerin yaklaşık yarısında
  `minifig_count=0` olacağından, model bu değişkeni büyük bir sıfır kütlesiyle birlikte
  öğrenecek — gerekirse `has_minifig` (bool) gibi ayrı bir ikili özellik de düşünülebilir.
